# DreamBooth Fine-Tuning (LoRA)
Fine-tune Stable Diffusion v1.5 on a subject using DreamBooth + LoRA with prior-preservation loss.

In [ ]:
!pip install -U "transformers<5.0.0" "huggingface_hub>=0.33.5" diffusers==0.33.0
!pip install git+https://github.com/openai/CLIP.git

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16 if device == "cuda" else torch.float32
print(f"Device: {device}  |  dtype: {dtype}")
if device == "cpu":
    print("WARNING: No GPU. Enable one via Runtime > Change runtime type > T4 GPU.")

In [ ]:
# ── Configure your subject here ─────────────────────────────
SUBJECT_NAME    = "killian"               # folder name under data/
CLASS_NAME      = "person" # folder name under data/prior/
CLASS_PROMPT    = "a person"
INSTANCE_PROMPT = f"a sks {CLASS_PROMPT}" # rare token binds subject to class

LORA_RANK  = 4  # LoRA bottleneck dimension — higher = more capacity, more memory
LORA_ALPHA = 4  # scaling factor; keep equal to rank unless you know why to change it

INSTANCE_DIR = f"data/{SUBJECT_NAME}"
PRIOR_DIR    = f"data/prior/{CLASS_NAME}"
# ─────────────────────────────────────────────────────────────
print(f"Instance  : {INSTANCE_DIR}  →  '{INSTANCE_PROMPT}'")
print(f"Prior     : {PRIOR_DIR}  →  '{CLASS_PROMPT}'")
print(f"LoRA rank : {LORA_RANK}  alpha: {LORA_ALPHA}")


## Upload source files
Select all 7 files from your local `code/` folder: `data.py`, `evaluate.py`, `generate_prior.py`, `inference.py`,  `metrics.py`, `model.py`, and `train.py`.

In [ ]:
from google.colab import files
uploaded = files.upload()
for fname in uploaded:
    print(f"loaded {fname}")

## Upload instance images

In [ ]:
import os
from google.colab import files

os.makedirs(INSTANCE_DIR, exist_ok=True)

uploaded = files.upload()
for fname, data in uploaded.items():
    with open(os.path.join(INSTANCE_DIR, fname), "wb") as f:
        f.write(data)

print(f"Uploaded {len(uploaded)} images to {INSTANCE_DIR}")

## Generate prior images
~15 min on T4 · ~5 min on A100

In [ ]:
from generate_prior import generate_prior_images

generate_prior_images(
    class_prompt=CLASS_PROMPT,
    output_dir=PRIOR_DIR,
    num_images=200,
    device=device,
    dtype=dtype,
)

## Train — LoRA
> T4-compatible. ~10–15 min on T4 · ~4–6 min on A100

In [ ]:
from train import training_loop

training_loop(
    instance_dir=INSTANCE_DIR,
    class_dir=PRIOR_DIR,
    instance_prompt=INSTANCE_PROMPT,
    class_prompt=CLASS_PROMPT,
    output_dir="checkpoints/lora",
    validation_dir="validation/lora",
    num_steps=800,
    device=device,
    dtype=dtype,
    lr=1e-4, 
    use_lora=True,
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
)


## Train — Full Fine-Tune
> **Requires A100** (Colab Pro+). Will OOM on T4.

~20–25 min on A100 (gradient checkpointing enabled)

In [ ]:
from train import training_loop

lora_only = False
training_loop(
    instance_dir=INSTANCE_DIR,
    class_dir=PRIOR_DIR,
    instance_prompt=INSTANCE_PROMPT,
    class_prompt=CLASS_PROMPT,
    output_dir="checkpoints/full",
    validation_dir="validation/full",
    num_steps=800,
    device=device,
    dtype=dtype,
    use_lora=False,
)

## Inference
Run both checkpoints on the same prompt so the outputs are directly comparable.

In [ ]:
from inference import run_inference

INFERENCE_PROMPT = f"{INSTANCE_PROMPT} eating pizza"  # change scene here

full_images = run_inference(
    checkpoint_dir="checkpoints/full/final",
    prompt=INFERENCE_PROMPT,
    output_dir="outputs/full",
    device=device,
    dtype=dtype,
    num_images=4,
)

lora_images = run_inference(
    checkpoint_dir="checkpoints/lora/final",
    prompt=INFERENCE_PROMPT,
    output_dir="outputs/lora",
    device=device,
    dtype=dtype,
    num_images=4,
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
)


## Visual comparison

In [ ]:
import matplotlib.pyplot as plt

if lora_only:
  fig, axes = plt.subplots(1, 4, figsize=(16, 8))
  for i, (ax, img) in enumerate(zip(axes, lora_images)):
      ax.imshow(img); ax.axis('off')
else:
  fig, axes = plt.subplots(2, 4, figsize=(16, 8))
  for i, (ax, img) in enumerate(zip(axes[0], full_images)):
      ax.imshow(img); ax.axis('off')
      ax.set_title('Full' if i == 0 else '')
  for i, (ax, img) in enumerate(zip(axes[1], lora_images)):
      ax.imshow(img); ax.axis('off')
      ax.set_title('LoRA' if i == 0 else '')
plt.suptitle(INFERENCE_PROMPT)
plt.tight_layout()
plt.show()

## Metrics comparison
CLIP-I and DINO measure subject fidelity (similarity to your real photos).
CLIP-T measures prompt fidelity (how well the scene matches the text).

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from metrics import compute_clip_i, compute_dino, compute_clip_t

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}

real_images = [
    Image.open(os.path.join(INSTANCE_DIR, f)).convert('RGB')
    for f in os.listdir(INSTANCE_DIR)
    if os.path.splitext(f)[1].lower() in IMAGE_EXTS
]

PAPER_REF = {'CLIP-I': 0.803, 'DINO': 0.668, 'CLIP-T': 0.305}

DESCRIPTIONS = {
    'CLIP-I': 'Subject fidelity — cosine similarity [0–1] between CLIP image embeddings of generated vs real photos',
    'DINO':   'Subject fidelity — cosine similarity [0–1] between DINO ViT features of generated vs real photos',
    'CLIP-T': 'Prompt fidelity — cosine similarity [0–1] between CLIP image embedding and CLIP text embedding of the prompt',
}

# ── Decide which models to evaluate ─────────────────────────
if lora_only:
    model_images = [('LoRA', lora_images)]
else:
    model_images = [
        ('Full fine-tune', full_images),
        ('LoRA', lora_images),
    ]

# ── Compute metrics ──────────────────────────────────────────
metrics = {}
for label, imgs in model_images:
    metrics[label] = {
        'CLIP-I': compute_clip_i(imgs, real_images),
        'DINO':   compute_dino(imgs, real_images),
        'CLIP-T': compute_clip_t(imgs, INFERENCE_PROMPT),
    }

# ── Printed table ───────────────────────────────────────────
if lora_only:
    print(f"{'Metric':<8} {'LoRA':>7} {'Paper*':>8}  Description")
else:
    print(f"{'Metric':<8} {'Full':>7} {'LoRA':>7} {'Paper*':>8}  Description")

print('-' * 80)

for m in ['CLIP-I', 'DINO', 'CLIP-T']:
    rv = PAPER_REF[m]

    if lora_only:
        lv = metrics['LoRA'][m]
        print(f"{m:<8} {lv:>7.3f} {rv:>8.3f}  {DESCRIPTIONS[m]}")
    else:
        fv = metrics['Full fine-tune'][m]
        lv = metrics['LoRA'][m]
        print(f"{m:<8} {fv:>7.3f} {lv:>7.3f} {rv:>8.3f}  {DESCRIPTIONS[m]}")

print()
print('* Paper ref = DreamBooth (Stable Diffusion), Ruiz et al. 2023')
print()

# ── Bar chart ───────────────────────────────────────────────
metric_names = ['CLIP-I', 'DINO', 'CLIP-T']

paper_vals = [PAPER_REF[m] for m in metric_names]

x = np.arange(len(metric_names))
w = 0.25

fig, ax = plt.subplots(figsize=(9, 5))

if lora_only:
    lora_vals = [metrics['LoRA'][m] for m in metric_names]

    ax.bar(x, lora_vals, w, label='LoRA', color='darkorange')
    ax.bar(x + w, paper_vals, w, label='Paper (SD)', color='gray', alpha=0.6)

else:
    full_vals = [metrics['Full fine-tune'][m] for m in metric_names]
    lora_vals = [metrics['LoRA'][m] for m in metric_names]

    ax.bar(x - w, full_vals, w, label='Full fine-tune', color='steelblue')
    ax.bar(x,     lora_vals, w, label='LoRA', color='darkorange')
    ax.bar(x + w, paper_vals, w, label='Paper (SD)', color='gray', alpha=0.6)

ax.set_xticks(x)
ax.set_xticklabels([
    'CLIP-I\n(subject fidelity)',
    'DINO\n(subject fidelity)',
    'CLIP-T\n(prompt fidelity)',
])
ax.set_ylabel('Cosine similarity [0–1]  (higher is better)')
ax.set_title('DreamBooth metric comparison vs paper baseline')
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()